In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from ocr_vs_vlm.results_final.shared.colors import (
    get_model_color, get_dataset_color, APPROACH_COLORS, AZURE, OPENAI, CLAUDE
)
from ocr_vs_vlm.results_final.shared.stats_utils import (
    bootstrap_ci, wilcoxon_test, cohens_d, effect_size_interpretation,
    compute_significance_matrix, bonferroni_correction
)
from ocr_vs_vlm.results_final.shared.viz_utils import (
    setup_plotly_template, create_metric_boxplot, create_heatmap, 
    create_radar_chart, create_pareto_chart
)
from ocr_vs_vlm.results_final.shared.data_loader import (
    load_all_data, extract_models_from_columns, compute_summary_statistics,
    PHASE_TO_APPROACH, load_pricing, compute_cost_performance
)

setup_plotly_template()
print("✓ Setup complete")

## 1. Load All Parsing Data

In [ ]:
PARSING_DATASETS = ['IAM_mini', 'ICDAR_mini', 'VOC2007']

try:
    df = load_all_data(datasets=PARSING_DATASETS, task_type='parsing')
    print(f"✓ Loaded {len(df)} total samples")
    print(f"  Datasets: {df['dataset'].unique().tolist()}")
    print(f"  Phases: {df['phase'].unique().tolist()}")
except Exception as e:
    print(f"Error: {e}")
    df = pd.DataFrame()

models = extract_models_from_columns(df) if len(df) > 0 else []
print(f"  Models: {models}")

In [ ]:
# Classify models as OCR or VLM
OCR_MODELS = ['azure_intelligence', 'mistral_document_ai', 'donut']
VLM_MODELS = ['gpt-5-mini', 'gpt-5-nano', 'claude_sonnet', 'claude_haiku']

def get_model_type(model):
    if model in OCR_MODELS:
        return 'OCR'
    elif model in VLM_MODELS:
        return 'VLM'
    return 'Unknown'

## 2. Global OCR vs VLM Comparison

In [ ]:
# Aggregate CER by model type across all datasets
global_stats = []

for dataset in PARSING_DATASETS:
    ds_df = df[df['dataset'] == dataset] if len(df) > 0 else pd.DataFrame()
    for model in models:
        col = f'cer_{model}'
        if col in ds_df.columns:
            scores = ds_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                global_stats.append({
                    'Dataset': dataset,
                    'Model': model,
                    'Type': get_model_type(model),
                    'CER': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi,
                    'N': len(scores)
                })

stats_df = pd.DataFrame(global_stats)
if len(stats_df) > 0:
    # Summary by type
    type_summary = stats_df.groupby('Type').agg({'CER': ['mean', 'std', 'count']}).round(4)
    display(Markdown("### Average CER by Model Type"))
    display(type_summary)

In [ ]:
# Visualization: OCR vs VLM across datasets
if len(stats_df) > 0:
    fig = go.Figure()
    
    for model_type in ['OCR', 'VLM']:
        type_data = stats_df[stats_df['Type'] == model_type]
        color = AZURE['primary'] if model_type == 'OCR' else OPENAI['primary']
        
        fig.add_trace(go.Box(
            y=type_data['CER'],
            x=type_data['Dataset'],
            name=model_type,
            marker_color=color,
            boxpoints='all',
            jitter=0.3
        ))
    
    fig.update_layout(
        title='Global Parsing: OCR vs VLM (CER by Dataset)',
        yaxis_title='Character Error Rate (Lower is Better)',
        boxmode='group',
        height=500
    )
    fig.show()

## 3. Statistical Significance Tests

In [ ]:
# Wilcoxon test: OCR vs VLM per dataset
print("📊 Statistical Comparison: OCR vs VLM")
print("=" * 70)

for dataset in PARSING_DATASETS:
    print(f"\n{dataset}:")
    ds_df = df[df['dataset'] == dataset] if len(df) > 0 else pd.DataFrame()
    
    # Collect OCR and VLM scores
    ocr_scores = []
    vlm_scores = []
    
    for model in models:
        col = f'cer_{model}'
        if col in ds_df.columns:
            scores = ds_df[col].dropna().values
            if get_model_type(model) == 'OCR':
                ocr_scores.extend(scores)
            elif get_model_type(model) == 'VLM':
                vlm_scores.extend(scores)
    
    if len(ocr_scores) > 0 and len(vlm_scores) > 0:
        ocr_mean = np.mean(ocr_scores)
        vlm_mean = np.mean(vlm_scores)
        
        # Paired test (taking min length)
        n = min(len(ocr_scores), len(vlm_scores))
        stat, p = wilcoxon_test(np.array(ocr_scores[:n]), np.array(vlm_scores[:n]))
        d = cohens_d(np.array(ocr_scores), np.array(vlm_scores))
        
        winner = 'OCR' if ocr_mean < vlm_mean else 'VLM'
        sig = '✓ Significant' if p < 0.05 else '✗ Not significant'
        
        print(f"   OCR CER: {ocr_mean:.4f} | VLM CER: {vlm_mean:.4f}")
        print(f"   Winner: {winner} | p={p:.4f} | d={d:.3f} ({effect_size_interpretation(d)})")
        print(f"   {sig}")

## 4. Model Ranking

In [ ]:
# Rank models by average CER across all datasets
if len(stats_df) > 0:
    ranking = stats_df.groupby('Model').agg({
        'CER': 'mean',
        'Type': 'first',
        'N': 'sum'
    }).sort_values('CER').reset_index()
    
    ranking['Rank'] = range(1, len(ranking) + 1)
    ranking = ranking[['Rank', 'Model', 'Type', 'CER', 'N']]
    
    display(Markdown("### Global Model Ranking (by Average CER)"))
    display(ranking.round(4))

In [ ]:
# Bar chart of ranking
if len(stats_df) > 0:
    ranking_sorted = ranking.sort_values('CER')
    
    colors = [AZURE['primary'] if t == 'OCR' else OPENAI['primary'] 
              for t in ranking_sorted['Type']]
    
    fig = go.Figure(go.Bar(
        x=ranking_sorted['CER'],
        y=ranking_sorted['Model'],
        orientation='h',
        marker_color=colors,
        text=ranking_sorted['CER'].round(4),
        textposition='outside'
    ))
    
    fig.update_layout(
        title='Global Parsing Model Ranking (CER - Lower is Better)',
        xaxis_title='Average CER',
        yaxis=dict(autorange='reversed'),
        height=400
    )
    fig.show()

## 5. Cost-Performance Analysis

In [ ]:
# Add cost estimates
if len(ranking) > 0:
    pricing = load_pricing()
    
    ranking['Cost_per_1K'] = ranking['Model'].apply(
        lambda m: pricing.get(m, {}).get('cost_per_1k', 
                  pricing.get(m, {}).get('input_per_1m', 0) * 1000 / 1_000_000)
    )
    
    # Performance = 1 - CER (so higher is better)
    ranking['Performance'] = 1 - ranking['CER']
    
    fig = create_pareto_chart(
        ranking[ranking['Cost_per_1K'] > 0],
        x_col='Cost_per_1K',
        y_col='Performance',
        label_col='Model',
        title='Parsing: Cost vs Performance (Pareto Frontier)',
        x_label='Cost per 1K samples ($)',
        y_label='Performance (1 - CER)'
    )
    fig.show()

## 6. Key Conclusions

In [ ]:
print("=" * 70)
print("GLOBAL PARSING COMPARISON: KEY CONCLUSIONS")
print("=" * 70)

if len(stats_df) > 0:
    ocr_avg = stats_df[stats_df['Type'] == 'OCR']['CER'].mean()
    vlm_avg = stats_df[stats_df['Type'] == 'VLM']['CER'].mean()
    
    print(f"\n📊 Overall Results:")
    print(f"   OCR Average CER: {ocr_avg:.4f}")
    print(f"   VLM Average CER: {vlm_avg:.4f}")
    print(f"   Winner: {'OCR' if ocr_avg < vlm_avg else 'VLM'}")
    
    if len(ranking) > 0:
        best = ranking.iloc[0]
        print(f"\n🏆 Best Overall Model: {best['Model']} ({best['Type']})")
        print(f"   CER: {best['CER']:.4f}")
    
    print("\n💡 Recommendations:")
    if ocr_avg < vlm_avg:
        print("   → For text extraction, OCR outperforms VLM on average")
        print("   → OCR is also more cost-effective for high-volume processing")
    else:
        print("   → VLM shows competitive or better performance than OCR")
        print("   → Consider VLM for complex document layouts")

print("\n" + "=" * 70)